# Baseline evaluation

Score the untouched Qwen3-8B on GSM8K and (optionally) MATH-500
before any training. These numbers are the reference we compare
every later stage against, so don't skip and don't change the
decoding settings between this notebook and notebook 06.

In [1]:
!pip install -q -U bitsandbytes transformers trl peft accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.5 MB/s eta 0:00:00


In [2]:
import torch
MODEL_NAME = 'Qwen/Qwen3-8B'
# Sanity load with 4-bit NF4 quantization just to confirm the model
# fits in memory and generates sensible text. We won't actually use
# 4-bit for the eval — lm-eval rejects load_in_4bit as a kwarg in
# recent versions, so we switch to bf16 below.


In [3]:
# One math problem as a smoke test. If this prints something that
# looks like reasoning followed by an answer, the model is loaded
# correctly. Keep enable_thinking=True so the model uses its
# <think>...</think> block — Qwen3 is trained for this and removing
# it hurts math accuracy.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa',  # use flash_attention_2 if flash-attn installed
)
model.eval()
print('Model loaded. VRAM used:', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded. VRAM used: 6.09 GB


In [4]:
# Free the model before running lm-eval. lm-eval will load its own
# copy of the model in a subprocess so we don't want two copies in
# VRAM at the same time.
messages = [
    {'role': 'system', 'content': 'Please reason step by step, and put your final answer within \\boxed{}.'},
    {'role': 'user', 'content': 'What is 15 * 47?'}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = tokenizer([text], return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512, temperature=0.6, top_p=0.95, do_sample=True)
response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(response)

<think>
Okay, so I need to figure out what 15 multiplied by 47 is. Let me think about how to approach this. I remember that multiplication can sometimes be broken down into easier parts. Maybe I can use the distributive property here? Let me recall, the distributive property says that a*(b + c) = a*b + a*c. So, if I can split 47 into two numbers that are easier to work with, maybe?

Let me try splitting 47 into 40 and 7. That seems reasonable because 40 + 7 is 47. So then, 15*47 would be 15*(40 + 7). Applying the distributive property, that becomes 15*40 + 15*7. Let me calculate each part separately.

First, 15*40. Hmm, 15 times 40. Well, 15*4 is 60, so 15*40 is 60*10, which is 600. Wait, let me check that again. 15*40: 10*40 is 400, and 5*40 is 200, so 400 + 200 is 600. Yeah, that's right.

Now the second part is 15*7. Let me calculate that. 10*7 is 70, and 5*7 is 35. Adding those together, 70 + 35 is 105. So 15*7 is 105.

Now, adding the two results together: 600 + 105. Let me do tha

In [5]:
# Run GSM8K with 8-shot chain-of-thought. --limit 100 keeps the
# run under 20 minutes on H100; bump to the full 1319 once you
# have time. --apply_chat_template is required for Qwen3.
# Note: no load_in_4bit here, just dtype=bfloat16.
del model
torch.cuda.empty_cache()
import gc; gc.collect()

6559

In [8]:
!pip install -q lm-eval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 135.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.4 MB/s eta 0:00:00


In [10]:
!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen3-8B,dtype=bfloat16 \
    --tasks gsm8k_cot \
    --num_fewshot 8 \
    --apply_chat_template \
    --limit 200 \
    --batch_size 4 \
    --output_path /content/results/baseline_gsm8k \
    --log_samples

2026-05-25:19:04:46 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-05-25:19:04:46 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-25:19:04:55 INFO     [_cli.run:388] Selected Tasks: ['gsm8k_cot']
2026-05-25:19:04:57 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-25:19:04:57 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3-8B', 'dtype': 'bfloat16'}
2026-05-25:19:05:02 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-25:19:05:04 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 399/399 [00:04<00:00, 91.60it/s] 
README.md: 7.93kB [00:00, 10.3MB/s]
main/train-00000-of-00001.parquet: 100% 2.31M/2.3

In [13]:
!pip install -q "lm-eval[math]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 20.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.


In [14]:
!lm_eval \
    --model hf \
    --model_args pretrained=Qwen/Qwen3-8B,dtype=bfloat16 \
    --tasks minerva_math \
    --num_fewshot 4 \
    --apply_chat_template \
    --limit 200 \
    --batch_size 4 \
    --output_path /content/results/baseline_math500

2026-05-25:19:25:18 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-05-25:19:25:18 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-05-25:19:25:28 INFO     [_cli.run:388] Selected Tasks: ['minerva_math']
2026-05-25:19:25:30 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-25:19:25:30 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3-8B', 'dtype': 'bfloat16'}
2026-05-25:19:25:35 INFO     [models.huggingface:286] Using device 'cuda:0'
2026-05-25:19:25:37 INFO     [models.huggingface:579] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 399/399 [00:04<00:00, 91.39it/s] 
README.md: 3.93kB [00:00, 13.1MB/s]
algebra/train-00000-of-00001.parquet: 100% 505

In [15]:
# Parse and display results
import json, glob

def load_result(path_glob):
    files = glob.glob(path_glob + '/**/*.json', recursive=True)
    if not files:
        print(f'No result file found at {path_glob}')
        return
    with open(files[0]) as f:
        data = json.load(f)
    for task, metrics in data.get('results', {}).items():
        print(f'{task}: {metrics}')

print('=== GSM8K ===')
load_result('/content/results/baseline_gsm8k')
print('=== MATH-500 ===')
load_result('/content/results/baseline_math500')

=== GSM8K ===
gsm8k_cot: {'name': 'gsm8k_cot', 'alias': 'gsm8k_cot', 'sample_len': 200, 'exact_match,strict-match': 0.02, 'exact_match_stderr,strict-match': 0.009924336870116691, 'exact_match,flexible-extract': 0.18, 'exact_match_stderr,flexible-extract': 0.027234326551496855}
=== MATH-500 ===
minerva_math_algebra: {'name': 'minerva_math_algebra', 'alias': 'minerva_math_algebra', 'sample_len': 200, 'exact_match,none': 0.0, 'exact_match_stderr,none': 0.0, 'math_verify,none': 0.095, 'math_verify_stderr,none': 0.020785455873744915}
minerva_math_counting_and_prob: {'name': 'minerva_math_counting_and_prob', 'alias': 'minerva_math_counting_and_prob', 'sample_len': 200, 'exact_match,none': 0.0, 'exact_match_stderr,none': 0.0, 'math_verify,none': 0.055, 'math_verify_stderr,none': 0.016161092305986408}
minerva_math_geometry: {'name': 'minerva_math_geometry', 'alias': 'minerva_math_geometry', 'sample_len': 200, 'exact_match,none': 0.0, 'exact_match_stderr,none': 0.0, 'math_verify,none': 0.055, '

In [16]:
# SAVE THESE NUMBERS — baseline for all future comparisons
baseline_results = {
    'model': MODEL_NAME,
    'stage': 'baseline',
    'gsm8k_acc': None,   # fill in from above
    'math500_acc': None, # fill in from above
    'notes': '4-bit NF4, lm-eval, 200-sample limit'
}
with open('/content/results/summary.json', 'w') as f:
    json.dump([baseline_results], f, indent=2)
print('Baseline results saved.')

Baseline results saved.


In [17]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/drive/MyDrive/llm_posttraining/results', exist_ok=True)
shutil.copytree('/content/results', '/content/drive/MyDrive/llm_posttraining/results', dirs_exist_ok=True)
print('Results backed up.')

Mounted at /content/drive
Results backed up.
